<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [ ]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [ ]:
# Display column names, data types, non-null counts, and missing-value counts
summary = pd.DataFrame({
    "DataType": df.dtypes.astype(str),
    "NonNullCount": df.notna().sum(),
    "MissingCount": df.isna().sum(),
})

display(summary)


<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [ ]:
# Basic descriptive statistics for numerical columns
display(df.describe().T)


### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [ ]:
# Inspect Country values and their frequencies.
# This lets us identify genuinely inconsistent entries instead of guessing.
country_counts = df["Country"].value_counts(dropna=False)
display(country_counts.head(30))

# Also check for leading/trailing whitespace in text values.
country_whitespace = df["Country"].dropna().astype(str)
country_whitespace_issues = country_whitespace[
    country_whitespace != country_whitespace.str.strip()
]

print(f"Country values with leading/trailing whitespace: {len(country_whitespace_issues)}")


<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [ ]:
# Standardize obvious formatting inconsistencies without inventing category mappings.
# strip() removes accidental leading/trailing spaces.
df["Country"] = df["Country"].astype("string").str.strip()

# Example mapping pattern:
# Only add a semantic mapping when inspection of the dataset confirms that
# two labels represent the same country/category.
country_mapping = {
    # "Old label": "Standard label",
}

df["Country"] = df["Country"].replace(country_mapping)

# Do the same basic formatting cleanup for EdLevel.
df["EdLevel"] = df["EdLevel"].astype("string").str.strip()

print("Country values after formatting standardization:")
display(df["Country"].value_counts(dropna=False).head(30))


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [ ]:
# One-hot encode Employment.
# dummy_na=True preserves missing Employment as its own indicator instead of
# silently hiding that information.
employment_dummies = pd.get_dummies(
    df["Employment"],
    prefix="Employment",
    dummy_na=True,
    dtype=int
)

# Add the encoded columns to the DataFrame.
df = pd.concat([df, employment_dummies], axis=1)

display(employment_dummies.head())


### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [ ]:
# Find columns with the highest number of missing values.
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .rename("MissingCount")
)

display(missing_summary[missing_summary > 0].to_frame())


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [ ]:
# Impute the numerical compensation column.
# Median is used because compensation data is typically skewed and is less
# sensitive to extreme values than the mean.
median_comp = df["ConvertedCompYearly"].median()

df["ConvertedCompYearly"] = df["ConvertedCompYearly"].fillna(median_comp)

print(f"Median used for ConvertedCompYearly imputation: {median_comp}")
print(
    "Remaining missing ConvertedCompYearly values:",
    df["ConvertedCompYearly"].isna().sum()
)


<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [ ]:
# Impute categorical RemoteWork values with the most frequent category.
remote_work_mode = df["RemoteWork"].mode(dropna=True)[0]

df["RemoteWork"] = df["RemoteWork"].fillna(remote_work_mode)

print(f"Most frequent RemoteWork value: {remote_work_mode}")
print(
    "Remaining missing RemoteWork values:",
    df["RemoteWork"].isna().sum()
)


### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [ ]:
# Min-Max scaling:
# (x - min) / (max - min)
comp_min = df["ConvertedCompYearly"].min()
comp_max = df["ConvertedCompYearly"].max()

if comp_max == comp_min:
    df["ConvertedCompYearly_MinMax"] = 0.0
else:
    df["ConvertedCompYearly_MinMax"] = (
        (df["ConvertedCompYearly"] - comp_min)
        / (comp_max - comp_min)
    )

display(
    df[["ConvertedCompYearly", "ConvertedCompYearly_MinMax"]].head()
)


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [ ]:
# Log transformation to reduce right-skew.
# log1p(x) calculates log(1 + x), which is safe for zero values.
df["ConvertedCompYearly_Log"] = np.log1p(df["ConvertedCompYearly"])

display(
    df[["ConvertedCompYearly", "ConvertedCompYearly_Log"]].head()
)


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [ ]:
# Convert YearsCodePro to numeric.
# The survey can contain non-numeric responses such as "Less than 1 year"
# and "More than 50 years", so handle those explicitly.
years = (
    df["YearsCodePro"]
      .astype("string")
      .str.strip()
      .replace({
          "Less than 1 year": "0",
          "More than 50 years": "51"
      })
)

years = pd.to_numeric(years, errors="coerce")

# Create ExperienceLevel using practical career-stage bands.
def experience_level(years):
    if pd.isna(years):
        return pd.NA
    elif years < 2:
        return "Beginner"
    elif years < 5:
        return "Intermediate"
    elif years < 10:
        return "Experienced"
    else:
        return "Senior"

df["ExperienceLevel"] = years.apply(experience_level).astype("string")

display(
    df[["YearsCodePro", "ExperienceLevel"]].head(20)
)


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
